In [0]:
df = spark.read.table("final_project.telecom_data.project_data")
df.show()



Bronze Layer

In [0]:
df = df.toDF(*[col.replace(' ', '_').replace(';', '_').replace('{', '_').replace('}', '_').replace('(', '_').replace(')', '_').replace(',', '_').replace('=', '_').replace('\n', '_').replace('\t', '_') for col in df.columns])
df.write.mode("overwrite").saveAsTable(
    "final_project.telecom_data.telecom_raw"
)

Silver Layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("final_project.telecom_data.telecom_raw")

Remove Duplicate Header Rows

In [0]:
df = df.filter(F.col("MSISDN_NUM") != "MSISDN_NUM")

Dropping Extra Columns

In [0]:
cols_to_drop = ["Unnamed: 28", "Unnamed: 29"]
df = df.drop(*[c for c in cols_to_drop if c in df.columns])

Rename Columns (Standard Naming)

In [0]:
df = df.withColumnRenamed("MSISDN_NUM", "msisdn") \
       .withColumnRenamed("Sales_Channel", "sales_channel") \
       .withColumnRenamed("Parent_Channel", "parent_channel") \
       .withColumnRenamed("Region", "region") \
       .withColumnRenamed("Sale_Date", "sale_date") \
       .withColumnRenamed("First_Active_Date", "first_active_date")

Handling of Null Values

In [0]:
df = df.fillna({
    "region": "Unknown",
    "sales_channel": "Unknown"
})

Data Quality Checks

In [0]:
sale_date_parsed = F.to_date(F.col("sale_date"), "M/d/yy")

df = df.filter(
    (F.length("msisdn") >= 10) &
    (sale_date_parsed.isNotNull()) &
    (sale_date_parsed <= F.current_date())
)

In [0]:
window_spec = Window.partitionBy("msisdn").orderBy(F.col("sale_date").desc())

df = df.withColumn("row_num", F.row_number().over(window_spec)) \
       .filter(F.col("row_num") == 1) \
       .drop("row_num")

In [0]:
df.show()

In [0]:
df = df.withColumn(
    "activity_status",
    F.when(F.col("Third_Month_Active").cast("int") == 1, "Active").otherwise("Inactive")
).withColumn(
    "recharge_flag",
    F.when(F.col("Thirty_Day_Recharge").cast("double") > 0, "Recharge_User").otherwise("No_Recharge")
).withColumn(
    "customer_lifetime_days",
    F.datediff(F.current_date(), F.to_date(F.col("first_active_date"), "M/d/yy"))
).withColumn(
    "total_recharge_90d",
    F.col("Thirty_Day_Recharge").cast("double") + F.col("sixty_day_recharge").cast("double") + F.col("ninety_day_recharge").cast("double")
)

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.telecom_silver")

In [0]:
df.count()

Gold Layer

In [0]:
df = spark.table("final_project.telecom_data.telecom_silver")

In [0]:
from pyspark.sql import functions as F

gold_quality_sales = df.groupBy("region", "sales_channel").agg(
    F.count("*").alias("total_sales"),
    F.sum("quality_sale_flag").alias("quality_sales"),
    (F.sum("quality_sale_flag") / F.count("*")).alias("quality_rate")
)

gold_quality_sales.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_quality_sales")

In [0]:
from pyspark.sql import functions as F

gold_active_region = df.groupBy("region").agg(
    F.countDistinct("msisdn").alias("total_users"),

    F.countDistinct(
        F.when(
            (F.col("Thirty_Day_Recharge") > 0) | 
            (F.col("7_Day_Recharge") > 0),
            F.col("msisdn")
        )
    ).alias("active_users")
)

gold_active_region = gold_active_region.withColumn(
    "active_percentage",
    (F.col("active_users") / F.col("total_users")) * 100
)

gold_active_region.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_active_customers_region")

In [0]:
gold_segments = df.withColumn(
    "customer_segment",
    F.when(F.col("total_recharge_90d") > 1000, "High Value")
     .when(F.col("total_recharge_90d") > 0, "Medium Value")
     .otherwise("Low Value")
)

gold_segments.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_customer_segments")

In [0]:
from pyspark.sql import functions as F

gold_active_parent_channel = df.groupBy("parent_channel").agg(
    
    # Total unique customers
    F.countDistinct("msisdn").alias("total_users"),

    # Active users based on recharge behavior
    F.countDistinct(
        F.when(
            (F.col("Thirty_Day_Recharge") > 0) | 
            (F.col("7_Day_Recharge") > 0),
            F.col("msisdn")
        )
    ).alias("active_users")
)

# Calculate percentage
gold_active_parent_channel = gold_active_parent_channel.withColumn(
    "active_percentage",
    (F.col("active_users") / F.col("total_users")) * 100
)

# Save table
gold_active_parent_channel.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_active_customers_parent_channel")

In [0]:
gold_segments.write.format("delta") \
    .mode("append") \
    .saveAsTable("final_project.telecom_data.gold_customer_segments")

In [0]:
from pyspark.sql import functions as F

gold_quality_sales_region = df.groupBy("region").agg(
    F.count("*").alias("total_sales"),
    F.sum("quality_sale_flag").alias("quality_sales"),
    (F.sum("quality_sale_flag") / F.count("*")).alias("quality_rate")
)

gold_quality_sales_region.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_quality_sales_region")

In [0]:
gold_quality_sales_parent_channel = df.groupBy("parent_channel", "region").agg(
    F.count("*").alias("total_sales"),
    F.sum("quality_sale_flag").alias("quality_sales"),
    (F.sum("quality_sale_flag") / F.count("*")).alias("quality_rate")
)

gold_quality_sales_parent_channel.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_quality_sales_parent_channel")

In [0]:
from pyspark.sql import functions as F

segments_by_region = gold_segments.groupBy("region").agg(
    F.sum(F.when(F.col("customer_segment") == "High Value", 1).otherwise(0)).alias("high_value"),
    F.sum(F.when(F.col("customer_segment") == "Medium Value", 1).otherwise(0)).alias("medium_value"),
    F.sum(F.when(F.col("customer_segment") == "Low Value", 1).otherwise(0)).alias("low_value")
)

segments_by_region.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_segments_region")

In [0]:
segments_by_parent_channel = gold_segments.groupBy("parent_channel").agg(
    F.sum(F.when(F.col("customer_segment") == "High Value", 1).otherwise(0)).alias("high_value"),
    F.sum(F.when(F.col("customer_segment") == "Medium Value", 1).otherwise(0)).alias("medium_value"),
    F.sum(F.when(F.col("customer_segment") == "Low Value", 1).otherwise(0)).alias("low_value")
)

segments_by_parent_channel.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.telecom_data.gold_segments_parent_channel")

ML MODEL

In [0]:
from pyspark.sql import functions as F

df = spark.table("final_project.telecom_data.telecom_silver")

df_ml = df.withColumn(
    "label",
    F.when(F.col("total_recharge_90d") > 1000, 1).otherwise(0)
)

In [0]:
df_ml = df_ml.select(
    "region",
    "sales_channel",
    "parent_channel",
    "Thirty_Day_Recharge",
    "7_Day_Recharge",
    "customer_lifetime_days",
    "label"
)

In [0]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(inputCol="region", outputCol="region_index", handleInvalid="keep"),
    StringIndexer(inputCol="sales_channel", outputCol="sales_channel_index", handleInvalid="keep"),
    StringIndexer(inputCol="parent_channel", outputCol="parent_channel_index", handleInvalid="keep")
]

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "region_index",
        "sales_channel_index",
        "parent_channel_index",
        "Thirty_Day_Recharge",
        "7_Day_Recharge",
        "customer_lifetime_days"
    ],
    outputCol="features"
)

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

try:
    del model
except NameError:
    pass

df_ml = df_ml.withColumn("Thirty_Day_Recharge", F.col("Thirty_Day_Recharge").cast("double")) \
             .withColumn("7_Day_Recharge", F.col("7_Day_Recharge").cast("double"))

lr = LogisticRegression(featuresCol="features", labelCol="label")

pipeline = Pipeline(stages=indexers + [assembler, lr])

model = pipeline.fit(df_ml)


In [0]:
predictions = model.transform(df_ml)

In [0]:
predictions.select(
    "msisdn",
    "prediction",
    "probability"
).write.format("delta") \
 .mode("overwrite") \
 .saveAsTable("final_project.telecom_data.gold_predictions")

In [0]:
try:
    del model
except NameError:
    pass

In [0]:
spark._client._cleanup_ml_cache()

train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
predictions = model.transform(test)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy = evaluator.evaluate(predictions)
print("AUC Score:", accuracy)